In [ ]:
# | default_exp preprocessing.ocr.audit_repair

In [ ]:
%load_ext autoreload
%autoreload 2

# OCR quality audit and repair queue

> Validate structured OCR layout sidecars without making OCR or network calls,
> and retain actionable region-level findings for a later repair pass.

The audit treats recorded OCR status as untrusted. It collects explicit failed,
incomplete, and non-trivially recovered regions, then independently checks
regions marked `completed` for empty or token-limited output, pathological
repetition, evaluator leakage, malformed table HTML, residual model tokens,
missing crop assets, invalid geometry, and duplicate region indexes.

The exported module is provider-independent for OCR outputs that follow the
structured schema-v2 `*.layout.json` shape used by the preprocessing notebooks.
Plain `.unlimited.json` metadata has no region records to inspect.


In [ ]:
# | export
import json
import re
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal, Sequence, cast


In [ ]:
# | export
def _layout_repetition_reason(content: str) -> str | None:
    fence_count = content.count("```")
    if fence_count >= 12:
        return f"contains {fence_count} Markdown fences"

    lines = [line.strip() for line in content.splitlines() if line.strip()]
    if lines:
        repeated_line, line_count = Counter(lines).most_common(1)[0]
        if line_count >= 8:
            return (
                f"repeats one {len(repeated_line)}-character line "
                f"{line_count} times"
            )

    compact = "".join(content.split())
    block_size = 32
    if len(compact) >= block_size * 2:
        blocks = Counter(
            compact[index : index + block_size]
            for index in range(len(compact) - block_size + 1)
        )
        _, block_count = blocks.most_common(1)[0]
        if block_count >= 8:
            return f"repeats a {block_size}-character block {block_count} times"
    return None


In [ ]:
# | export
OCRQualityIssueKind = Literal[
    "failed", "incomplete", "recovered", "suspicious_completed"
]


@dataclass(frozen=True)
class OCRRegionQualityIssue:
    """One layout region that should be reviewed or repaired."""

    page_number: int
    region_index: int
    label: str
    task_type: str
    recorded_status: str
    issue_kind: OCRQualityIssueKind
    reasons: tuple[str, ...]
    error: str | None
    bbox: tuple[int, int, int, int] | None
    asset_path: Path | None
    content_preview: str


@dataclass(frozen=True)
class OCRFileQualityReport:
    """Quality findings for one structured OCR layout sidecar."""

    sidecar_path: Path
    source_path: Path | None
    document_status: str
    pages_total: int
    partial_pages: tuple[int, ...]
    file_reasons: tuple[str, ...]
    region_issues: tuple[OCRRegionQualityIssue, ...]

    @property
    def needs_repair(self) -> bool:
        """Whether this sidecar contains any file- or region-level issue."""
        return bool(self.file_reasons or self.region_issues)


_OCR_EVALUATOR_MARKERS = (
    "ground truth image",
    "according to rule",
    "provided ocr content",
    "the ocr result",
    "ocr result must",
    "violating the rule",
)
_OCR_CONTROL_TOKENS = (
    "<|ref|>",
    "<|/ref|>",
    "<|det|>",
    "<|/det|>",
    "<|image_pad|>",
    "<|vision_start|>",
    "<|vision_end|>",
)


def _layout_evaluator_leakage_reason(content: str) -> str | None:
    lowered = content.casefold()
    matches = [
        marker for marker in _OCR_EVALUATOR_MARKERS if marker in lowered
    ]
    if len(matches) < 2:
        return None
    return "contains OCR evaluator commentary: " + ", ".join(matches[:3])


def _layout_table_repetition_reason(content: str) -> str | None:
    lines = [line.strip() for line in content.splitlines() if line.strip()]
    if not lines:
        return None
    repeated_line, line_count = Counter(lines).most_common(1)[0]
    if line_count < 25:
        return None
    return (
        f"repeats one {len(repeated_line)}-character table line "
        f"{line_count} times"
    )


def _layout_placeholder_flood_reason(content: str) -> str | None:
    placeholders = re.findall(
        r"(?i)(?:\[(?:image|img)\]|<img(?:\s[^>]*)?>|<page(?:\s[^>]*)?>)",
        content,
    )
    if len(placeholders) < 20:
        return None
    return f"contains {len(placeholders)} repeated image placeholders"


def _quality_bbox(value: object) -> tuple[int, int, int, int] | None:
    if not isinstance(value, Sequence) or isinstance(value, (str, bytes)):
        return None
    try:
        coordinates = tuple(int(coordinate) for coordinate in value)
    except (TypeError, ValueError):
        return None
    if len(coordinates) != 4:
        return None
    return cast(tuple[int, int, int, int], coordinates)


def _quality_path(
    value: object,
    sidecar_path: Path,
    *,
    prefix: object = None,
) -> Path | None:
    if not isinstance(value, str) or not value.strip():
        return None
    path = Path(value)
    if path.is_absolute():
        return path
    root = sidecar_path.parent
    if isinstance(prefix, str) and prefix.strip():
        prefix_path = Path(prefix)
        root = prefix_path if prefix_path.is_absolute() else root / prefix_path
    return root / path


def _quality_preview(content: str, max_characters: int = 240) -> str:
    return re.sub(r"\s+", " ", content).strip()[:max_characters]


def _completed_layout_quality_reasons(
    record: dict[str, Any],
    *,
    sidecar_path: Path,
    asset_prefix: object = None,
) -> tuple[str, ...]:
    content = str(record.get("content") or "")
    task_type = str(record.get("task_type") or "unknown")
    status = str(record.get("status") or "missing")
    region_description = (
        "completed region" if status == "completed" else "region"
    )
    reasons: list[str] = []
    if task_type != "figure" and not content.strip():
        reasons.append(f"{region_description} has no usable OCR content")
    if record.get("error"):
        reasons.append(
            f"{region_description} records an error: {record['error']}"
        )
    finish_reason = record.get("finish_reason")
    if finish_reason == "length":
        reasons.append("model output stopped at the token limit")
    elif finish_reason not in {None, "stop"}:
        reasons.append(f"model output has abnormal finish reason {finish_reason!r}")
    if status == "completed" and record.get("recovery"):
        reasons.append(f"completed region records recovery: {record['recovery']}")

    repetition_reason = (
        _layout_table_repetition_reason(content)
        if task_type == "table"
        else _layout_repetition_reason(content)
    )
    if repetition_reason is not None:
        reasons.append(f"degenerate OCR output {repetition_reason}")
    placeholder_reason = _layout_placeholder_flood_reason(content)
    if placeholder_reason is not None:
        reasons.append(placeholder_reason)
    leakage_reason = _layout_evaluator_leakage_reason(content)
    if leakage_reason is not None:
        reasons.append(leakage_reason)

    lowered = content.casefold()
    if lowered.count("<table") != lowered.count("</table>"):
        reasons.append("contains an unbalanced HTML table")
    leaked_tokens = [token for token in _OCR_CONTROL_TOKENS if token in content]
    if leaked_tokens:
        reasons.append(
            "contains unremoved model control tokens: "
            + ", ".join(leaked_tokens[:3])
        )

    asset_path = _quality_path(
        record.get("asset"), sidecar_path, prefix=asset_prefix
    )
    if asset_path is not None and not asset_path.exists():
        reasons.append(f"referenced region asset is missing: {asset_path}")
    return tuple(dict.fromkeys(reasons))


def _layout_region_quality_issue(
    record: dict[str, Any],
    *,
    page_number: int,
    fallback_index: int,
    sidecar_path: Path,
    asset_prefix: object = None,
    duplicate_index: bool = False,
    page_size: tuple[int, int] | None = None,
) -> OCRRegionQualityIssue | None:
    status = str(record.get("status") or "missing")
    label = str(record.get("label") or "unknown")
    task_type = str(record.get("task_type") or "unknown")
    error = str(record["error"]) if record.get("error") else None
    recovery = (
        str(record["recovery"]) if record.get("recovery") else None
    )
    asset_path = _quality_path(
        record.get("asset"), sidecar_path, prefix=asset_prefix
    )
    content = str(record.get("content") or "")
    try:
        region_index = int(record.get("index", fallback_index))
    except (TypeError, ValueError):
        region_index = fallback_index
        invalid_index = True
    else:
        invalid_index = region_index < 1
        if invalid_index:
            region_index = fallback_index
    bbox = _quality_bbox(record.get("bbox"))
    reasons: list[str] = []
    if invalid_index:
        reasons.append("region index is missing or invalid")
    if duplicate_index:
        reasons.append(f"page contains duplicate region index {region_index}")
    if bbox is None:
        reasons.append("region bbox is missing or invalid")
    else:
        left, top, right, bottom = bbox
        if left < 0 or top < 0 or right <= left or bottom <= top:
            reasons.append(f"region bbox has invalid coordinates: {bbox}")
        elif (
            page_size is not None
            and (right > page_size[0] or bottom > page_size[1])
        ):
            reasons.append(
                f"region bbox {bbox} exceeds page size {page_size}"
            )
    issue_kind: OCRQualityIssueKind

    if status == "failed":
        issue_kind = "failed"
        reasons.append(error or "region status is failed without an error message")
    elif status == "recovered":
        if (
            not reasons
            and label in {"footer", "header", "number"}
            and not content.strip()
            and recovery is not None
            and "decorative or empty" in recovery.casefold()
            and (asset_path is None or asset_path.exists())
        ):
            return None
        issue_kind = "recovered"
        reasons.append(recovery or "region required OCR recovery")
        reasons.extend(
            _completed_layout_quality_reasons(
                record,
                sidecar_path=sidecar_path,
                asset_prefix=asset_prefix,
            )
        )
    elif status == "preserved":
        if (
            task_type == "figure"
            and not error
            and asset_path is not None
            and asset_path.exists()
            and not reasons
        ):
            return None
        issue_kind = "incomplete"
        reasons.append(
            error
            or recovery
            or "region was preserved without usable OCR text or crop asset"
        )
    elif status == "completed":
        reasons.extend(
            _completed_layout_quality_reasons(
                record,
                sidecar_path=sidecar_path,
                asset_prefix=asset_prefix,
            )
        )
        if not reasons:
            return None
        issue_kind = "suspicious_completed"
    else:
        issue_kind = "incomplete"
        reasons.append(f"region status is {status!r}, not completed")
        if error:
            reasons.append(error)

    if asset_path is not None and not asset_path.exists():
        reasons.append(f"referenced region asset is missing: {asset_path}")
    elif status == "preserved" and asset_path is None:
        reasons.append("preserved region has no referenced crop asset")
    return OCRRegionQualityIssue(
        page_number=page_number,
        region_index=region_index,
        label=label,
        task_type=task_type,
        recorded_status=status,
        issue_kind=issue_kind,
        reasons=tuple(dict.fromkeys(reasons)),
        error=error,
        bbox=bbox,
        asset_path=asset_path,
        content_preview=_quality_preview(content),
    )


def audit_layout_ocr_file(sidecar_path: str | Path) -> OCRFileQualityReport:
    """Audit one ``*.layout.json`` sidecar without making OCR calls."""
    selected_sidecar = Path(sidecar_path).expanduser().resolve()
    try:
        layout = json.loads(selected_sidecar.read_text(encoding="utf-8"))
    except Exception as error:
        return OCRFileQualityReport(
            sidecar_path=selected_sidecar,
            source_path=None,
            document_status="invalid",
            pages_total=0,
            partial_pages=(),
            file_reasons=(
                f"cannot read layout sidecar: {type(error).__name__}: {error}",
            ),
            region_issues=(),
        )

    if not isinstance(layout, dict):
        return OCRFileQualityReport(
            sidecar_path=selected_sidecar,
            source_path=None,
            document_status="invalid",
            pages_total=0,
            partial_pages=(),
            file_reasons=("layout sidecar root must be a JSON object",),
            region_issues=(),
        )

    source_path = _quality_path(layout.get("source"), selected_sidecar)
    asset_prefix = layout.get("asset_prefix")
    document_status = str(layout.get("status") or "missing")
    pages = layout.get("pages")
    file_reasons: list[str] = []
    partial_pages: list[int] = []
    region_issues: list[OCRRegionQualityIssue] = []

    if document_status == "partial":
        file_reasons.append("document status is partial")
    elif document_status != "processed":
        file_reasons.append(f"document status is {document_status!r}")
    if not isinstance(pages, list):
        pages = []
        file_reasons.append("layout sidecar has no valid pages list")

    try:
        pages_total = int(layout.get("pages_total", len(pages)))
    except (TypeError, ValueError):
        pages_total = len(pages)
        file_reasons.append("pages_total is missing or invalid")
    if pages_total != len(pages):
        file_reasons.append(
            f"pages_total is {pages_total}, but the sidecar has {len(pages)} page slots"
        )

    for page_position, page in enumerate(pages, start=1):
        if not isinstance(page, dict):
            file_reasons.append(f"page slot {page_position} is missing or invalid")
            continue
        try:
            page_number = int(page.get("page_number", page_position))
        except (TypeError, ValueError):
            page_number = page_position
            file_reasons.append(f"page slot {page_position} has an invalid number")
        page_status = str(page.get("status") or "missing")
        if page_status == "partial":
            partial_pages.append(page_number)
        elif page_status != "completed":
            file_reasons.append(f"page {page_number} status is {page_status!r}")
        if page.get("error"):
            file_reasons.append(f"page {page_number}: {page['error']}")
        try:
            page_width = int(page["width"])
            page_height = int(page["height"])
        except (KeyError, TypeError, ValueError):
            page_size = None
        else:
            page_size = (page_width, page_height)

        regions = page.get("regions")
        if not isinstance(regions, list):
            file_reasons.append(f"page {page_number} has no valid regions list")
            continue
        seen_region_indices: set[int] = set()
        for region_position, record in enumerate(regions, start=1):
            if not isinstance(record, dict):
                file_reasons.append(
                    f"page {page_number} region slot {region_position} is invalid"
                )
                continue
            try:
                recorded_index = int(record.get("index"))
            except (TypeError, ValueError):
                duplicate_index = False
            else:
                duplicate_index = recorded_index in seen_region_indices
                seen_region_indices.add(recorded_index)
            issue = _layout_region_quality_issue(
                record,
                page_number=page_number,
                fallback_index=region_position,
                sidecar_path=selected_sidecar,
                asset_prefix=asset_prefix,
                duplicate_index=duplicate_index,
                page_size=page_size,
            )
            if issue is not None:
                region_issues.append(issue)

    if partial_pages:
        rendered_pages = ", ".join(str(number) for number in partial_pages)
        file_reasons.append(f"partial pages: {rendered_pages}")
    return OCRFileQualityReport(
        sidecar_path=selected_sidecar,
        source_path=source_path,
        document_status=document_status,
        pages_total=pages_total,
        partial_pages=tuple(partial_pages),
        file_reasons=tuple(dict.fromkeys(file_reasons)),
        region_issues=tuple(
            sorted(
                region_issues,
                key=lambda issue: (issue.page_number, issue.region_index),
            )
        ),
    )


def _canonical_layout_sidecars(
    root: Path,
    *,
    include_partial_checkpoints: bool,
) -> list[Path]:
    sidecars = list(root.rglob("*.layout.json"))
    if not include_partial_checkpoints:
        selected: list[Path] = []
        for sidecar in sidecars:
            if not sidecar.name.endswith(".partial.layout.json"):
                selected.append(sidecar)
                continue
            final_name = (
                sidecar.name.removesuffix(".partial.layout.json")
                + ".layout.json"
            )
            if not sidecar.with_name(final_name).exists():
                selected.append(sidecar)
        sidecars = selected
    return sorted(sidecars, key=lambda path: str(path).casefold())


def audit_layout_ocr_results(
    root: str | Path,
    *,
    include_partial_checkpoints: bool = False,
) -> list[OCRFileQualityReport]:
    """Audit every structured OCR sidecar beneath ``root``."""
    selected_root = Path(root).expanduser().resolve()
    if not selected_root.exists():
        raise FileNotFoundError(f"OCR quality-audit root does not exist: {selected_root}")
    if selected_root.is_file():
        sidecars = [selected_root]
    else:
        sidecars = _canonical_layout_sidecars(
            selected_root,
            include_partial_checkpoints=include_partial_checkpoints,
        )
    if not sidecars:
        warnings.warn(
            f"No structured *.layout.json OCR results found in {selected_root}",
            stacklevel=2,
        )
    return [audit_layout_ocr_file(sidecar) for sidecar in sidecars]


def collect_ocr_repair_queue(
    root: str | Path,
    *,
    include_partial_checkpoints: bool = False,
) -> list[OCRFileQualityReport]:
    """Return only files with explicit or inferred OCR quality issues."""
    return [
        report
        for report in audit_layout_ocr_results(
            root,
            include_partial_checkpoints=include_partial_checkpoints,
        )
        if report.needs_repair
    ]

In [ ]:
# | export
OCRFixScope = Literal["file", "region"]
OCRFixAction = Literal[
    "rebuild_sidecar",
    "reconcile_manifest",
    "resume_document",
    "rerun_layout_detection",
    "regenerate_crop",
    "split_and_retry",
    "discard_and_reprocess",
    "adjust_crop_and_retry",
    "review_or_discard",
    "retry_region",
    "resume_region",
    "verify_recovery",
    "manual_review",
]


@dataclass(frozen=True)
class OCRFixProposal:
    """A read-only, actionable proposal for one OCR quality issue."""

    sidecar_path: Path
    source_path: Path | None
    scope: OCRFixScope
    action: OCRFixAction
    summary: str
    steps: tuple[str, ...]
    reasons: tuple[str, ...]
    page_number: int | None
    region_index: int | None
    label: str | None
    task_type: str | None
    bbox: tuple[int, int, int, int] | None
    asset_path: Path | None


def _file_fix_strategy(
    reason: str,
) -> tuple[OCRFixAction, str, tuple[str, ...]]:
    lowered = reason.casefold()
    if "pages_total" in lowered:
        return (
            "reconcile_manifest",
            "Reconcile the sidecar page manifest with the source document.",
            (
                "Compare pages_total and page slots with the source page count.",
                "Regenerate missing or inconsistent page records from the source.",
                "Publish the corrected manifest and re-audit it.",
            ),
        )
    if any(
        marker in lowered
        for marker in (
            "cannot read layout sidecar",
            "sidecar root",
            "no valid pages list",
            "page slot",
            "region slot",
        )
    ):
        return (
            "rebuild_sidecar",
            "Recover or rebuild the structured layout sidecar before repairing regions.",
            (
                "Restore the newest valid partial checkpoint when one exists.",
                "Otherwise rerun layout detection from the source document.",
                "Atomically publish and re-audit the replacement sidecar.",
            ),
        )
    if (
        any(
            marker in lowered
            for marker in (
                "document status",
                "partial pages",
                "page status",
                "status is 'processing'",
                "status is 'pending'",
            )
        )
        or (
            lowered.startswith("page ")
            and any(
                marker in lowered
                for marker in (" status is ", "has no valid regions list", ": ")
            )
        )
    ):
        return (
            "resume_document",
            "Resume the incomplete document from its durable checkpoint.",
            (
                "Resolve unfinished pages and regions from the sidecar.",
                "Resume with the recorded OCR and layout settings.",
                "Preserve completed regions, publish, and re-audit.",
            ),
        )
    return (
        "manual_review",
        "Inspect the file-level inconsistency before attempting region repair.",
        (
            "Compare the source, Markdown, final sidecar, and partial checkpoint.",
            "Record the corrected document state and run the audit again.",
        ),
    )


def _region_fix_strategy(
    issue: OCRRegionQualityIssue,
) -> tuple[OCRFixAction, str, tuple[str, ...]]:
    lowered = " ".join(issue.reasons).casefold()
    if any(
        marker in lowered
        for marker in ("duplicate region index", "region index", "bbox")
    ):
        return (
            "rerun_layout_detection",
            "Repair region identity or geometry before retrying OCR.",
            (
                "Re-render the source page at the recorded DPI.",
                "Rerun layout detection and reconcile neighboring region indexes.",
                "Replace the bbox/index, regenerate its crop, and re-audit.",
            ),
        )
    if (
        "asset is missing" in lowered
        or "no referenced crop asset" in lowered
    ):
        return (
            "regenerate_crop",
            "Regenerate the missing region crop before OCR retry.",
            (
                "Render the source page and crop the recorded bbox.",
                "Verify the crop is non-empty and stored beside the sidecar.",
                "Retry OCR on the regenerated crop and validate the response.",
            ),
        )
    if any(
        marker in lowered
        for marker in (
            "token limit",
            "output-token limit",
            "unbalanced html table",
        )
    ):
        return (
            "split_and_retry",
            "Split the oversized or truncated region and retry it in smaller units.",
            (
                "Prefer aligned native PDF text when a reliable text layer exists.",
                "Otherwise tile the crop with overlap along rows or reading order.",
                "OCR each tile, stitch the result, and validate structure and repetition.",
            ),
        )
    if any(
        marker in lowered
        for marker in (
            "degenerate",
            "repeated image placeholder",
            "evaluator commentary",
            "control token",
            "pathological",
            "abnormal finish reason",
        )
    ):
        return (
            "discard_and_reprocess",
            "Quarantine the untrustworthy OCR content and reprocess the original crop.",
            (
                "Keep the current content only as provenance; exclude it from retrieval.",
                "Retry from the crop or aligned native PDF text, preferably with an alternate model.",
                "Accept only output that passes repetition, markup, and leakage checks.",
            ),
        )
    if any(
        marker in lowered
        for marker in (
            "no usable ocr content",
            "empty response",
            "empty after post-processing",
        )
    ):
        if issue.label in {"footer", "header", "number"}:
            return (
                "review_or_discard",
                "Decide whether this empty decorative region needs OCR at all.",
                (
                    "Inspect the crop and its neighboring reading-order regions.",
                    "If decorative or truly empty, record an intentional discard.",
                    "Otherwise expand or enhance the crop and retry OCR.",
                ),
            )
        return (
            "adjust_crop_and_retry",
            "Improve the crop signal and retry the empty OCR region.",
            (
                "Inspect whether the region is meaningful rather than decorative.",
                "Expand the bbox slightly and adjust resolution or contrast.",
                "Retry OCR; preserve the crop for manual review if it remains empty.",
            ),
        )
    if issue.issue_kind == "incomplete":
        return (
            "resume_region",
            "Resume this pending or interrupted region from the checkpoint.",
            (
                "Confirm the region crop and recorded settings are still compatible.",
                "Retry only this region without overwriting completed neighbors.",
                "Publish the updated sidecar and re-audit the page.",
            ),
        )
    if issue.issue_kind == "failed":
        return (
            "retry_region",
            "Retry the failed region while retaining the original error as provenance.",
            (
                "Verify service availability, crop readability, and request settings.",
                "Retry with bounded backoff and the same region identity.",
                "Validate the replacement before clearing the failure.",
            ),
        )
    if issue.issue_kind == "recovered":
        return (
            "verify_recovery",
            "Verify the recovered prefix or fallback before accepting it as final.",
            (
                "Compare recovered content with the crop and available native PDF text.",
                "Accept it only when semantic and structural checks agree.",
                "Otherwise requeue the original crop for OCR.",
            ),
        )
    if issue.issue_kind == "suspicious_completed":
        return (
            "discard_and_reprocess",
            "Treat the suspicious completed output as untrusted and reprocess it.",
            (
                "Exclude the current content from downstream retrieval.",
                "Reprocess from the original crop or native PDF text.",
                "Require a clean audit before restoring completed status.",
            ),
        )
    return (
        "manual_review",
        "Inspect the region and choose a repair method manually.",
        ("Compare the crop, OCR content, and neighboring regions.",),
    )


def propose_ocr_repairs(
    ocr_repair_queue: Sequence[OCRFileQualityReport],
) -> list[OCRFixProposal]:
    """Return one deterministic fix proposal per file reason and region issue."""
    proposals: list[OCRFixProposal] = []
    for report in sorted(
        ocr_repair_queue, key=lambda item: str(item.sidecar_path).casefold()
    ):
        for reason in report.file_reasons:
            action, summary, steps = _file_fix_strategy(reason)
            proposals.append(
                OCRFixProposal(
                    sidecar_path=report.sidecar_path,
                    source_path=report.source_path,
                    scope="file",
                    action=action,
                    summary=summary,
                    steps=steps,
                    reasons=(reason,),
                    page_number=None,
                    region_index=None,
                    label=None,
                    task_type=None,
                    bbox=None,
                    asset_path=None,
                )
            )
        for issue in report.region_issues:
            action, summary, steps = _region_fix_strategy(issue)
            proposals.append(
                OCRFixProposal(
                    sidecar_path=report.sidecar_path,
                    source_path=report.source_path,
                    scope="region",
                    action=action,
                    summary=summary,
                    steps=steps,
                    reasons=issue.reasons,
                    page_number=issue.page_number,
                    region_index=issue.region_index,
                    label=issue.label,
                    task_type=issue.task_type,
                    bbox=issue.bbox,
                    asset_path=issue.asset_path,
                )
            )
    return proposals


def build_ocr_repair_plan(
    root: str | Path,
    *,
    include_partial_checkpoints: bool = False,
) -> list[OCRFixProposal]:
    """Audit a result tree and return proposals for its repair queue."""
    return propose_ocr_repairs(
        collect_ocr_repair_queue(
            root,
            include_partial_checkpoints=include_partial_checkpoints,
        )
    )

## Audit a result tree

Other notebooks can import the generated module directly:

```python
from ribosome.preprocessing.ocr.audit_repair import (
    audit_layout_ocr_file,
    audit_layout_ocr_results,
    build_ocr_repair_plan,
    collect_ocr_repair_queue,
    propose_ocr_repairs,
)
```

`audit_layout_ocr_results()` returns one deterministic report per canonical
sidecar, including clean files. A transient `.partial.layout.json` checkpoint
is suppressed when its published sibling exists, while orphan checkpoints
remain auditable. `collect_ocr_repair_queue()` filters this list to reports
whose file or region findings need repair or review.

Each region finding retains its page/index, bounding box, crop path, compact
content preview, declared status, issue kind, and concrete reasons. Intentionally
preserved figures and discarded empty decorative headers, footers, or page
numbers are not queued.


In [ ]:
# | notest
from ribosome.preprocessing.ocr.utils import find_project_root

PROJ_ROOT = find_project_root()
OCR_QUALITY_ROOT = PROJ_ROOT / "assets" / "unlimited_ocr_mit_coding"

ocr_quality_reports = audit_layout_ocr_results(OCR_QUALITY_ROOT)
ocr_repair_queue = [
    report for report in ocr_quality_reports if report.needs_repair
]
[
    {
        "sidecar": str(report.sidecar_path),
        "status": report.document_status,
        "partial_pages": report.partial_pages,
        "file_reasons": report.file_reasons,
        "repair_regions": len(report.region_issues),
    }
    for report in ocr_repair_queue
]


## Propose fixes for every queued issue

Proposal generation is deterministic and advisory-only: it creates no OCR
client, network request, subprocess, or file write. It emits one proposal for
each `file_reason` and one consolidated proposal for each region issue, so no
finding in `ocr_repair_queue` is silently dropped. Every proposal retains its
source/sidecar location, region coordinates and crop when applicable, original
reasons, a stable action code, a short summary, and ordered repair steps.

The routing order protects prerequisites: repair invalid layout identity or
geometry first, regenerate missing crops next, split token-limited or malformed
regions, quarantine contaminated output, then handle empty, incomplete, failed,
or recovered OCR. Unknown findings always receive a manual-review proposal.
Use `build_ocr_repair_plan(root)` when the result tree has not already been
audited.

In [ ]:
# | notest
ocr_fix_proposals = propose_ocr_repairs(ocr_repair_queue)
ocr_fix_proposal_rows = [
    {
        "sidecar": str(proposal.sidecar_path),
        "scope": proposal.scope,
        "page": proposal.page_number,
        "region": proposal.region_index,
        "action": proposal.action,
        "summary": proposal.summary,
        "steps": proposal.steps,
        "reasons": proposal.reasons,
    }
    for proposal in ocr_fix_proposals
]
# Keep the complete list above; render only a compact notebook preview.
ocr_fix_proposal_rows[:25]

## Tests

The regression test uses generated sidecars and crop assets in a temporary
directory. It makes no network calls and writes nothing to the repository.


In [ ]:
# | hide
from tempfile import TemporaryDirectory

from fastcore.test import test_eq


def test_ocr_quality_audit_and_repair_queue():
    def region(
        index,
        *,
        status="completed",
        content="valid OCR text",
        label="text",
        task_type="text",
        error=None,
        recovery=None,
        finish_reason="stop",
        asset=None,
        bbox=(10, 20, 110, 120),
    ):
        return {
            "index": index,
            "label": label,
            "task_type": task_type,
            "status": status,
            "content": content,
            "error": error,
            "recovery": recovery,
            "finish_reason": finish_reason,
            "asset": asset,
            "bbox": list(bbox),
        }

    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        asset_root = root / "checkpoint-work"
        asset_root.mkdir()
        (asset_root / "repeat.png").write_bytes(b"crop")
        (asset_root / "figure.png").write_bytes(b"figure")

        good = {
            "status": "processed",
            "source": "good.pdf",
            "pages_total": 1,
            "pages": [
                {
                    "page_number": 1,
                    "status": "completed",
                    "error": None,
                    "regions": [region(1)],
                }
            ],
        }
        problem = {
            "status": "partial",
            "source": "problem.pdf",
            "asset_prefix": "checkpoint-work",
            "pages_total": 1,
            "pages": [
                {
                    "page_number": 1,
                    "status": "partial",
                    "error": None,
                    "regions": [
                        region(
                            1,
                            status="failed",
                            content="",
                            error="ValueError: output token limit",
                            finish_reason=None,
                        ),
                        region(
                            2,
                            label="table",
                            task_type="table",
                            content="\n".join(["[Image]"] * 25),
                            asset="repeat.png",
                        ),
                        region(
                            3,
                            label="table",
                            task_type="table",
                            content=(
                                "The Ground Truth image differs. "
                                "According to Rule 2, the provided OCR content "
                                "must be rejected."
                            ),
                        ),
                        region(4, content=""),
                        region(
                            5,
                            status="recovered",
                            recovery="Recovered a complete prefix",
                            finish_reason="length",
                        ),
                        region(
                            6,
                            status="preserved",
                            content="",
                            finish_reason=None,
                        ),
                        region(
                            7,
                            status="preserved",
                            content="",
                            label="image",
                            task_type="figure",
                            finish_reason=None,
                            asset="figure.png",
                        ),
                        region(8),
                        region(8, content="duplicate region index"),
                        region(9, status="processing"),
                        region(
                            10,
                            label="table",
                            task_type="table",
                            content="<table><tr><td>truncated",
                        ),
                        region(
                            11,
                            content="text <|det|>[0, 0, 1, 1]<|/det|>",
                            asset="missing.png",
                        ),
                        region(12, bbox=(20, 20, 10, 120)),
                        region(
                            13,
                            status="recovered",
                            content="",
                            label="number",
                            recovery=(
                                "Discarded a degenerate Unlimited-OCR response "
                                "for a decorative or empty number region"
                            ),
                            finish_reason="stop",
                        ),
                    ],
                }
            ],
        }
        orphan_checkpoint = {
            "status": "processing",
            "source": "orphan.pdf",
            "pages_total": 0,
            "pages": [],
        }

        (root / "good.layout.json").write_text(
            json.dumps(good), encoding="utf-8"
        )
        (root / "problem.layout.json").write_text(
            json.dumps(problem), encoding="utf-8"
        )
        (root / "problem.partial.layout.json").write_text(
            json.dumps(problem), encoding="utf-8"
        )
        (root / "orphan.partial.layout.json").write_text(
            json.dumps(orphan_checkpoint), encoding="utf-8"
        )
        (root / "invalid.layout.json").write_text("{", encoding="utf-8")

        reports = audit_layout_ocr_results(root)
        test_eq(
            [report.sidecar_path.name for report in reports],
            [
                "good.layout.json",
                "invalid.layout.json",
                "orphan.partial.layout.json",
                "problem.layout.json",
            ],
        )
        report_by_name = {report.sidecar_path.name: report for report in reports}
        assert not report_by_name["good.layout.json"].needs_repair
        assert report_by_name["invalid.layout.json"].needs_repair
        assert report_by_name["orphan.partial.layout.json"].needs_repair

        problem_report = report_by_name["problem.layout.json"]
        test_eq(problem_report.partial_pages, (1,))
        issues = {issue.region_index: issue for issue in problem_report.region_issues}
        test_eq(sorted(issues), [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12])
        test_eq(issues[1].issue_kind, "failed")
        test_eq(issues[1].error, "ValueError: output token limit")
        assert "repeats one 7-character table line 25 times" in issues[2].reasons[0]
        assert any("image placeholders" in reason for reason in issues[2].reasons)
        test_eq(issues[2].asset_path, asset_root / "repeat.png")
        assert any("evaluator commentary" in reason for reason in issues[3].reasons)
        test_eq(issues[4].issue_kind, "suspicious_completed")
        test_eq(issues[5].issue_kind, "recovered")
        test_eq(issues[6].issue_kind, "incomplete")
        assert any("duplicate region index" in reason for reason in issues[8].reasons)
        test_eq(issues[9].issue_kind, "incomplete")
        assert any("unbalanced HTML table" in reason for reason in issues[10].reasons)
        assert any("control tokens" in reason for reason in issues[11].reasons)
        assert any("asset is missing" in reason for reason in issues[11].reasons)
        assert any("invalid coordinates" in reason for reason in issues[12].reasons)

        repair_queue = collect_ocr_repair_queue(root)
        test_eq(
            [report.sidecar_path.name for report in repair_queue],
            [
                "invalid.layout.json",
                "orphan.partial.layout.json",
                "problem.layout.json",
            ],
        )
        test_eq(
            len(
                audit_layout_ocr_results(
                    root, include_partial_checkpoints=True
                )
            ),
            5,
        )


test_ocr_quality_audit_and_repair_queue()


In [ ]:
# | hide
def test_ocr_fix_proposals():
    def issue(
        index,
        issue_kind,
        reasons,
        *,
        label="text",
        task_type="text",
        bbox=(10, 20, 110, 120),
        asset_path=None,
    ):
        return OCRRegionQualityIssue(
            page_number=1,
            region_index=index,
            label=label,
            task_type=task_type,
            recorded_status=(
                "completed"
                if issue_kind == "suspicious_completed"
                else issue_kind
            ),
            issue_kind=issue_kind,
            reasons=tuple(reasons),
            error=None,
            bbox=bbox,
            asset_path=asset_path,
            content_preview="preview",
        )

    region_issues = (
        issue(1, "suspicious_completed", ("region bbox is invalid",), bbox=None),
        issue(
            2,
            "incomplete",
            ("referenced region asset is missing",),
            asset_path=Path("missing.png"),
        ),
        issue(
            3,
            "failed",
            ("output-token limit",),
            label="table",
            task_type="table",
        ),
        issue(
            4,
            "suspicious_completed",
            ("contains OCR evaluator commentary",),
        ),
        issue(
            5,
            "failed",
            ("empty after post-processing",),
            label="number",
        ),
        issue(
            6,
            "failed",
            ("completed region has no usable OCR content",),
        ),
        issue(7, "failed", ("APIConnectionError: unavailable",)),
        issue(8, "incomplete", ("region status is 'processing'",)),
        issue(9, "recovered", ("Recovered a complete prefix",)),
        issue(
            10,
            "suspicious_completed",
            ("unclassified completed-region concern",),
        ),
    )
    file_reasons = (
        "cannot read layout sidecar: invalid JSON",
        "pages_total is 3, but the sidecar has 2 page slots",
        "document status is partial",
        "unknown file-level concern",
    )
    report = OCRFileQualityReport(
        sidecar_path=Path("problem.layout.json"),
        source_path=Path("problem.pdf"),
        document_status="partial",
        pages_total=1,
        partial_pages=(1,),
        file_reasons=file_reasons,
        region_issues=region_issues,
    )

    proposals = propose_ocr_repairs([report])
    test_eq(len(proposals), len(file_reasons) + len(region_issues))
    test_eq(propose_ocr_repairs([report]), proposals)
    assert all(proposal.summary and proposal.steps for proposal in proposals)

    file_actions = {
        proposal.reasons[0]: proposal.action
        for proposal in proposals
        if proposal.scope == "file"
    }
    test_eq(
        [file_actions[reason] for reason in file_reasons],
        [
            "rebuild_sidecar",
            "reconcile_manifest",
            "resume_document",
            "manual_review",
        ],
    )
    region_actions = {
        proposal.region_index: proposal.action
        for proposal in proposals
        if proposal.scope == "region"
    }
    test_eq(
        region_actions,
        {
            1: "rerun_layout_detection",
            2: "regenerate_crop",
            3: "split_and_retry",
            4: "discard_and_reprocess",
            5: "review_or_discard",
            6: "adjust_crop_and_retry",
            7: "retry_region",
            8: "resume_region",
            9: "verify_recovery",
            10: "discard_and_reprocess",
        },
    )
    connection_proposal = next(
        proposal for proposal in proposals if proposal.region_index == 7
    )
    assert "backoff" in " ".join(connection_proposal.steps).casefold()


test_ocr_fix_proposals()
